# HS Code 수출 데이터의 보냉재 2사 실적 선행성 검증**korea_export_fs_leadlag_v1**- 대상 종목: 동성화인텍(A033500), 한국카본(A017960)- 수출 품목: HS 390931(폴리머릭 MDI, **투입물**), HS 390950(폴리우레탄, **산출물**)- 수출 데이터: `investar.korea_monthly_trade_data` (2017-01 ~ 2026-07, 월별)- 재무 데이터: `investar.korea_fs_data_from_DG` (DataGuide, **천원** 단위, 분기말 영업일)- DB 접근: **읽기 전용(SELECT)**## 검증 설계상 주의점1. **HS 390931은 두 회사의 투입원가다.** MDI는 초저온 보냉재용 폴리우레탄 폼의 원재료다.   따라서 단가 상승 → 마진 압박이라는 **음(−)의 상관**이 이론적으로 기대되는 부호다.2. **HS 390950은 산출물이지만 매출 경로를 포착하지 못한다.** 두 회사의 보냉재는 국내   조선소에 납품되므로 국내 매출이고, 최종 수출 시에는 HS 8901(선박)로 계상된다.3. **영업이익 YoY는 사용하지 않는다.** 저(低)base 분기(예: 동성화인텍 2020Q4 −2.5억원)   때문에 YoY가 −2,156%까지 폭발해 Pearson 상관이 극단값 1~2개에 지배된다.   마진 타깃은 **영업이익률의 전년동기 차분(OPM Δ4, %p)** 을 쓴다.4. **중첩 YoY의 자기상관을 보정한다.** 4분기 중첩차분이라 계열 자기상관이 높다   (390931 단가 rho1 = 0.82). 명목 n이 아니라 유효표본 n_eff로 p를 재계산한다.

---# PART 1 — 환경 및 Control Panel

In [ ]:
import sys, warningssys.path.insert(0, r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy")import numpy as npimport pandas as pdimport matplotlib as mplimport matplotlib.pyplot as pltfrom scipy import statsfrom DATA.config import get_db_info, get_enginewarnings.filterwarnings("ignore")pd.set_option("display.width", 200)pd.set_option("display.max_columns", 60)mpl.rcParams["font.family"] = "Malgun Gothic"mpl.rcParams["axes.unicode_minus"] = Falsempl.rcParams["figure.dpi"] = 110ENGINE = get_engine(get_db_info())   # 읽기 전용으로만 사용print("engine ready:", ENGINE.url.database)

In [ ]:
# ============================ CONTROL PANEL ============================# 수출EXPORT_TABLE = "korea_monthly_trade_data"       # root_hs_code / indicator / value (월별)HS_CODES     = {"390931": "MDI(투입물)", "390950": "폴리우레탄(산출물)"}EXP_IND      = ["expDlr", "expWgt"]             # 금액(천USD) / 중량(kg)# 재무FS_TABLE = "korea_fs_data_from_DG"TICKERS  = {"A033500": "동성화인텍", "A017960": "한국카본"}ITEMS    = {"M000904001": "매출액", "M000906001": "영업이익"}FS_UNIT_DIVISOR = 1e8 / 1000                    # 천원 -> 억원# 분석 구간START_Q = pd.Period("2017Q1", freq="Q")END_Q   = pd.Period("2026Q2", freq="Q")         # 수출 원천이 2026-07까지 -> 2026Q3 불완전MAX_LAG = 4                                     # 수출(t-lag) vs 재무(t)MIN_OBS = 8                                     # 상관 계산 최소 표본# 진단 임계ADJ_KEEP_OK   = 0.55   # 인접 lag 유지율 >= 0.55 -> '완만'(신뢰)ADJ_KEEP_FAIL = 0.35   # < 0.35 -> '급락'(우연 의심)SPEARMAN_GAP  = 0.25   # |Pearson - Spearman| > 0.25 -> 극단값 의심# =======================================================================

---# PART 2 — 함수 정의

In [ ]:
def load_export_quarterly(engine, table, hs_codes, indicators, end_q):    """월별 수출 -> 분기 집계. 3개월 완비 분기만 채택. 단가 = 합계금액 / 합계중량(물량가중)."""    ph = ", ".join(f"%(h{i})s" for i in range(len(hs_codes)))    pi = ", ".join(f"%(i{j})s" for j in range(len(indicators)))    q = (f"SELECT date, root_hs_code, indicator, value FROM {table} "         f"WHERE root_hs_code IN ({ph}) AND indicator IN ({pi})")    params = {f"h{i}": h for i, h in enumerate(hs_codes)}    params.update({f"i{j}": v for j, v in enumerate(indicators)})    m = pd.read_sql(q, engine, params=params)    m["q"] = pd.to_datetime(m["date"]).dt.to_period("Q")    n_month = m[m.indicator == indicators[0]].groupby(["root_hs_code", "q"]).size().rename("n_month")    agg = m.groupby(["root_hs_code", "q", "indicator"])["value"].sum().unstack("indicator").join(n_month)    agg = agg[(agg["n_month"] == 3) & (agg.index.get_level_values("q") <= end_q)]    agg["unit_price"] = agg["expDlr"] / agg["expWgt"]    out = agg[["expDlr", "expWgt", "unit_price"]].unstack("root_hs_code")    out.columns = [f"{hs}_{var}" for var, hs in out.columns]    return out.sort_index()def load_fs_quarterly(engine, table, tickers, items, divisor):    """DataGuide 분기 재무. 0값은 NaN 치환, 천원 -> 억원."""    ph = ", ".join(f"%(t{i})s" for i in range(len(tickers)))    pi = ", ".join(f"%(c{j})s" for j in range(len(items)))    q = (f"SELECT date, ticker, item_code, value FROM {table} "         f"WHERE ticker IN ({ph}) AND item_code IN ({pi})")    params = {f"t{i}": t for i, t in enumerate(tickers)}    params.update({f"c{j}": c for j, c in enumerate(items)})    f = pd.read_sql(q, engine, params=params)    f["q"] = pd.to_datetime(f["date"]).dt.to_period("Q")   # 분기말 영업일 -> 회계분기 귀속    f["value"] = f["value"].replace(0, np.nan) / divisor    f["col"] = f["ticker"].map(tickers) + "_" + f["item_code"].map(items)    return f.pivot_table(index="q", columns="col", values="value").sort_index()def build_panel(exp, fs, start_q, end_q):    """레벨 + YoY(%) + OPM / OPM Δ4 를 한 패널로."""    lv = exp.join(fs, how="outer").sort_index()    lv = lv[(lv.index >= start_q) & (lv.index <= end_q)]    for name in sorted(set(c.split("_")[0] for c in fs.columns)):        rev, op = f"{name}_매출액", f"{name}_영업이익"        if rev in lv and op in lv:            lv[f"{name}_OPM"] = lv[op] / lv[rev] * 100            lv[f"{name}_OPM_d4"] = lv[f"{name}_OPM"].diff(4)    base = [c for c in lv.columns if not c.endswith(("_OPM", "_OPM_d4"))]    yoy = lv[base].pct_change(4) * 100    yoy.columns = [c + "_yoy" for c in yoy.columns]    return lv.join(yoy)def eff_n(n, rho_x, rho_y):    """중첩 YoY 자기상관 보정 유효표본 (Quenouille/Bartlett 근사)."""    if np.isnan(rho_x) or np.isnan(rho_y):        return float(n)    return max(n * (1 - rho_x * rho_y) / (1 + rho_x * rho_y), 4.0)def crosscorr_profile(panel, xcol, ycol, max_lag=None, min_obs=None):    """lag 0..max_lag 에 대해 Pearson/Spearman/n 계산. 수출(t-lag) vs 재무(t)."""    max_lag = MAX_LAG if max_lag is None else max_lag    min_obs = MIN_OBS if min_obs is None else min_obs    rows = []    for lag in range(max_lag + 1):        d = pd.concat([panel[xcol].shift(lag).rename("x"), panel[ycol].rename("y")], axis=1).dropna()        if len(d) < min_obs:            rows.append({"lag": lag, "n": len(d), "r": np.nan, "p": np.nan, "rs": np.nan, "ps": np.nan})            continue        r, p = stats.pearsonr(d["x"], d["y"])        rs, ps = stats.spearmanr(d["x"], d["y"])        rows.append({"lag": lag, "n": len(d), "r": r, "p": p, "rs": rs, "ps": ps})    return pd.DataFrame(rows).set_index("lag")def diagnose(panel, prof, xcol, ycol):    """최적 lag + 강건성 진단(인접 lag 유지율, Pearson-Spearman 괴리, n_eff 보정 p)."""    if prof["r"].isna().all():        return None    L = int(prof["r"].abs().idxmax())    r, rs, n = prof.loc[L, "r"], prof.loc[L, "rs"], int(prof.loc[L, "n"])    nb = [prof["r"].get(L - 1, np.nan), prof["r"].get(L + 1, np.nan)]    nb = [abs(v) for v in nb if not pd.isna(v)]    keep = float(np.mean(nb)) / abs(r) if nb and r else np.nan    ne = eff_n(n, panel[xcol].autocorr(1), panel[ycol].autocorr(1))    t = r * np.sqrt((ne - 2) / (1 - r ** 2))    p_adj = 2 * (1 - stats.t.cdf(abs(t), ne - 2))    gap = abs(r - rs)    if gap > SPEARMAN_GAP:      shape = "극단값의존"    elif keep >= ADJ_KEEP_OK:   shape = "완만"    elif keep < ADJ_KEEP_FAIL:  shape = "급락"    else:                       shape = "중간"    return {"best_lag": L, "n": n, "r": r, "p": prof.loc[L, "p"], "rs": rs, "ps": prof.loc[L, "ps"],            "인접유지율": keep, "n_eff": ne, "p_adj": p_adj, "P-S괴리": gap, "판정": shape}def scan_all(panel, hs_codes, xvars, firms, targets):    """전체 조합 스캔 -> 진단 포함 요약표."""    out = []    for hs in hs_codes:        for xk, xn in xvars.items():            xcol = f"{hs}_{xn}_yoy"            for fm in firms:                for tk, ysuf in targets.items():                    ycol = f"{fm}_{ysuf}"                    if xcol not in panel or ycol not in panel:                        continue                    prof = crosscorr_profile(panel, xcol, ycol)                    d = diagnose(panel, prof, xcol, ycol)                    if d is None:                        continue                    shape = " ".join(f"{v:+.2f}" if not pd.isna(v) else "  .  " for v in prof["r"])                    out.append({"HS": hs, "수출변수": xk, "종목": fm, "타깃": tk,                                "프로파일": shape, **d})    return pd.DataFrame(out)def plot_lag_heatmap(panel, hs_codes, xvars, firms, targets, ax=None):    cells, rows, cols = {}, [], []    for hs in hs_codes:        for xk, xn in xvars.items():            rlab = f"{hs[-4:]} {xk}"            if rlab not in rows:                rows.append(rlab)            for fm in firms:                for tk, ysuf in targets.items():                    clab = f"{fm[:2]} {tk}"                    if clab not in cols:                        cols.append(clab)                    xcol, ycol = f"{hs}_{xn}_yoy", f"{fm}_{ysuf}"                    if xcol in panel and ycol in panel:                        cells[(rlab, clab)] = crosscorr_profile(panel, xcol, ycol)["r"].values    W = MAX_LAG + 1    M = np.full((len(rows), len(cols) * W), np.nan)    for i, r in enumerate(rows):        for j, c in enumerate(cols):            v = cells.get((r, c))            if v is not None:                M[i, j * W:(j + 1) * W] = v    if ax is None:        _, ax = plt.subplots(figsize=(13, 4.6))    im = ax.imshow(M, cmap="RdBu", vmin=-0.8, vmax=0.8, aspect="auto")    ax.set_yticks(range(len(rows)))    ax.set_yticklabels(rows, fontsize=9)    ax.set_xticks(range(M.shape[1]))    ax.set_xticklabels([f"L{l}" for _ in cols for l in range(W)], fontsize=7)    for j, c in enumerate(cols):        ax.text(j * W + MAX_LAG / 2, -0.95, c, ha="center", va="bottom", fontsize=9,                clip_on=False)        if j:            ax.axvline(j * W - 0.5, color="white", lw=2.2)    for i in range(M.shape[0]):        for j in range(M.shape[1]):            if not np.isnan(M[i, j]):                ax.text(j, i, f"{M[i, j]:.2f}".replace("0.", "."), ha="center", va="center",                        fontsize=6.4, color="white" if abs(M[i, j]) > 0.45 else "#444444")    ax.set_title("YoY 교차상관 (수출 t-lag -> 재무 t)", pad=42, fontsize=11)    plt.colorbar(im, ax=ax, shrink=0.8, label="Pearson r")    return axdef plot_pair(panel, xcol, ycol, lag, xlab, ylab, figsize=(12, 3.8)):    """선정 조합: 시계열 중첩(수출 부호반전) + 산점도."""    d = pd.concat([panel[xcol].shift(lag).rename("x"), panel[ycol].rename("y")], axis=1).dropna()    r, p = stats.pearsonr(d["x"], d["y"])    rs, _ = stats.spearmanr(d["x"], d["y"])    fig, (a1, a2) = plt.subplots(1, 2, figsize=figsize, gridspec_kw={"width_ratios": [1.7, 1]})    t = d.index.to_timestamp()    a1.plot(t, d["y"], color="#eb6834", lw=2, label=ylab)    a1.plot(t, -d["x"], color="#2a78d6", lw=2, ls="--", label=f"{xlab} (부호반전)")    a1.axhline(0, color="#aaaaaa", lw=0.8)    a1.legend(fontsize=8)    a1.grid(alpha=0.25)    a1.set_title(f"시계열 중첩 · lag {lag}분기", fontsize=10)    a2.scatter(d["x"], d["y"], s=34, color="#eb6834", edgecolor="white", zorder=3)    b, a0 = np.polyfit(d["x"], d["y"], 1)    xs = np.linspace(d["x"].min(), d["x"].max(), 10)    a2.plot(xs, a0 + b * xs, color="#888888", ls="--", lw=1.4)    a2.set_xlabel(xlab, fontsize=8)    a2.set_ylabel(ylab, fontsize=8)    a2.grid(alpha=0.25)    a2.set_title(f"n={len(d)}  r={r:+.3f} (p={p:.4f})  rho={rs:+.3f}", fontsize=10)    fig.tight_layout()    return fig

---# PART 3 — 실행

In [ ]:
# 3-1. 적재exp = load_export_quarterly(ENGINE, EXPORT_TABLE, list(HS_CODES), EXP_IND, END_Q)fs  = load_fs_quarterly(ENGINE, FS_TABLE, TICKERS, ITEMS, FS_UNIT_DIVISOR)panel = build_panel(exp, fs, START_Q, END_Q)print("수출 분기:", exp.index.min(), "~", exp.index.max(), f"({len(exp)}분기)")print("재무 분기:", fs.index.min(), "~", fs.index.max())print("패널 shape:", panel.shape)panel.tail(6).round(1)

In [ ]:
# 3-2. 커버리지 점검cov = pd.DataFrame({    "n_obs": panel.notna().sum(),    "first": [panel[c].first_valid_index() for c in panel],    "last":  [panel[c].last_valid_index()  for c in panel],    "rho1":  [panel[c].autocorr(1) for c in panel],})cov[cov.index.str.endswith("_yoy") | cov.index.str.endswith("_OPM_d4")].round(3)

In [ ]:
# 3-3. STEP 2 — 점유율 구조 변화 스크리닝: log(분기매출 / 분기수출금액) 추세screen = []for fm in TICKERS.values():    for hs in HS_CODES:        s = np.log(panel[f"{fm}_매출액"] / panel[f"{hs}_expDlr"]).dropna()        t = np.arange(len(s), dtype=float)        lr = stats.linregress(t, s.values)        screen.append({"종목": fm, "HS": hs, "n": len(s),                       "기울기/분기": lr.slope, "연율(%/년)": (np.exp(lr.slope * 4) - 1) * 100,                       "R2": lr.rvalue ** 2, "p": lr.pvalue,                       "구조변화": "심각" if lr.rvalue ** 2 > 0.5 else ("경미" if lr.pvalue < 0.05 else "없음")})screen = pd.DataFrame(screen)display(screen.round(4))fig, ax = plt.subplots(figsize=(11, 3.6))for fm, color in zip(TICKERS.values(), ["#2a78d6", "#eb6834"]):    for hs, ls in zip(HS_CODES, ["-", "--"]):        s = np.log(panel[f"{fm}_매출액"] / panel[f"{hs}_expDlr"]).dropna()        ax.plot(s.index.to_timestamp(), np.exp(s - s.iloc[0]) * 100,                color=color, ls=ls, lw=2, label=f"{fm} / {hs}")ax.axhline(100, color="#aaaaaa", lw=0.8)ax.legend(fontsize=8, ncol=2)ax.grid(alpha=0.25)ax.set_ylabel("지수 (첫 분기 = 100)")ax.set_title("매출 / 수출금액 비율 추이", fontsize=11)plt.tight_layout(); plt.show()

In [ ]:
# 3-4. STEP 3 — 전체 조합 교차상관 스캔XVARS   = {"금액": "expDlr", "중량": "expWgt", "단가": "unit_price"}FIRMS   = list(TICKERS.values())TARGETS = {"매출": "매출액_yoy", "마진": "OPM_d4"}    # 영업이익 YoY 대신 OPM Δ4scan = scan_all(panel, list(HS_CODES), XVARS, FIRMS, TARGETS)scan = scan.reindex(scan["r"].abs().sort_values(ascending=False).index)display(scan[["HS", "수출변수", "종목", "타깃", "best_lag", "n", "r", "rs", "p_adj",              "인접유지율", "n_eff", "판정", "프로파일"]].round(3))print(f"\n검정 수: {len(scan)}  |  Bonferroni 임계 p < {0.05 / len(scan):.5f}")print("보정 p_adj < 0.05 통과:", int((scan['p_adj'] < 0.05).sum()), "건")print("그 중 판정='완만':", int(((scan['p_adj'] < 0.05) & (scan['판정'] == '완만')).sum()), "건")

In [ ]:
# 3-5. 참고 — 영업이익 YoY 를 썼을 때 무엇이 깨지는지 (기각 근거)bad = []for hs in HS_CODES:    for xk, xn in XVARS.items():        for fm in FIRMS:            prof = crosscorr_profile(panel, f"{hs}_{xn}_yoy", f"{fm}_영업이익_yoy")            if prof["r"].isna().all():                continue            L = int(prof["r"].abs().idxmax())            bad.append({"HS": hs, "수출변수": xk, "종목": fm, "lag": L, "n": int(prof.loc[L, "n"]),                        "Pearson": prof.loc[L, "r"], "Spearman": prof.loc[L, "rs"],                        "괴리": abs(prof.loc[L, "r"] - prof.loc[L, "rs"])})bad = pd.DataFrame(bad).sort_values("괴리", ascending=False)display(bad.round(3).head(8))print("영업이익 YoY 분포 (극단값 확인):")display(panel[[f"{f}_영업이익_yoy" for f in FIRMS]].describe().round(1))

---# PART 4 — 결과## 4-1. 교차상관 히트맵

In [ ]:
plot_lag_heatmap(panel, list(HS_CODES), XVARS, FIRMS, TARGETS)plt.tight_layout(); plt.show()

## 4-2. 물량(중량) vs 단가 — 어느 쪽이 실적과 더 연관되는가

In [ ]:
cmp = scan[scan["수출변수"].isin(["중량", "단가"])].copy()cmp["abs_rs"] = cmp["rs"].abs()print("■ 전체 평균 |Spearman rho|")display(cmp.pivot_table(index="수출변수", columns="타깃", values="abs_rs", aggfunc="mean").round(3))print("■ 종목별")display(cmp.pivot_table(index=["종목", "수출변수"], columns="타깃", values="abs_rs", aggfunc="mean").round(3))print("■ 조합 상세")display(cmp[["HS", "종목", "수출변수", "타깃", "best_lag", "n", "rs", "ps", "판정", "프로파일"]]        .sort_values(["종목", "타깃", "수출변수"]).round(3))

**가설 대비 결과**| 가설 | 예측 | 실제 ||---|---|---|| 물량(중량) → 매출 | 중량이 매출과 강하게 연관 | ✗ 평균 abs(rho) 0.358, 전 조합에서 가장 약함 || 단가 → 마진 | 단가가 마진과 강하게 연관 | △ 평균 abs(rho) 0.411, 부호는 음(−)으로 투입원가 가설과 일치 || — | — | **단가가 매출과 더 강함** (평균 abs(rho) 0.513) |물량 채널은 어느 타깃에서도 살아나지 않는다. 한국 MDI·PU 수출 물량은 중국 향 범용수요가 좌우하는데, 보냉재 2사의 생산량은 LNG선 건조 스케줄이 정하므로 두 물량이공유하는 사이클이 없다. 반면 단가는 두 산업이 같은 원료(MDI) 가격에 노출되므로채널이 존재한다.

## 4-3. 선행지표 후보 — 한국카본 한정

In [ ]:
CANDIDATES = [    ("390931_expDlr_yoy",     "한국카본_매출액_yoy", 3, "MDI 수출금액 YoY (t-3)", "한국카본 매출 YoY (t)"),    ("390931_unit_price_yoy", "한국카본_OPM_d4",     0, "MDI 단가 YoY (t)",       "한국카본 OPM d4 (%p)"),]for xcol, ycol, lag, xlab, ylab in CANDIDATES:    prof = crosscorr_profile(panel, xcol, ycol)    print(f"\n[{xlab} -> {ylab}]")    display(prof.round(4))    plot_pair(panel, xcol, ycol, lag, xlab, ylab)    plt.show()

## 4-4. 결론### 후보 조합과 강건성 (4-5 결과 반영)| 지표 | 타깃 | lag | 전체구간 r / rho | 2021Q4~22Q2 제외 r / rho | 판정 ||---|---|---|---|---|---|| HS 390931 단가 YoY | 한국카본 OPM Δ4 | **0 (동행)** | −0.574 / −0.532 | **−0.572 / −0.462** | ✅ 안정 || HS 390931 수출금액 YoY | 한국카본 매출 YoY | 3 (선행) | −0.600 / −0.669 | **−0.356 / −0.516** | △ 취약 |**결론이 4-5에서 뒤집힌다.** 처음에는 "MDI 수출금액이 3분기 선행"이 가장 강한 조합으로보였으나, 2021Q4~2022Q2(중국 MDI 수요 급등 + LNG 발주 공백이 겹친 구간) 3개 분기를빼면 Pearson이 −0.600 → −0.356으로 무너지고 p=0.063으로 유의성을 잃는다.이 조합의 상관은 특정 3개 분기가 만든 것이다.반대로 **단가 → 마진(동행)은 −0.574 → −0.572로 사실상 불변**이다. 특정 구간에기대지 않는 유일한 관계이며, 부호도 투입원가 가설(MDI 가격 상승 → 마진 압박)과정확히 일치한다. 실전에서 쓸 수 있는 건 이쪽이다. 다만 lag 0이 최강이므로**선행지표가 아니라 동행지표**다.### 사용 불가 판정| 조합 | Pearson | Spearman | 기각 사유 ||---|---|---|---|| 390950 중량 → 동성화인텍 영업이익 (L2) | −0.774 | **−0.066** | 극단값 1~2개가 상관을 생성 || 390950 금액 → 동성화인텍 영업이익 (L2) | −0.761 | **−0.157** | 동일 + lag 프로파일 급락(L1 −0.13, L3 −0.17) || 390931 단가 → 동성화인텍 영업이익 (L2) | −0.608 | −0.278 | 극단값 의존 || 레벨 기준 전반 (120건 중 81건 p<0.05) | — | — | 공통 시간추세(spurious) |### 해석상 유보 — 상관의 부호가 왜 전부 음수인가히트맵 대부분의 셀이 음수다. "MDI 수출이 늘면 3분기 뒤 보냉재 매출이 준다"는 인과로읽으면 안 된다. 금액·중량 기반 상관의 대부분은 **2021Q4~2022Q2** 구간이 만든다.이때 MDI 수출은 +143% YoY로 급증(중국 건설·가전 수요)한 반면 한국카본 매출은−24~−33%였다(2020~21년 LNG 발주 공백의 시차 반영). 두 산업 사이클의 **위상차**를상관계수가 포착한 것이지, 한쪽이 다른 쪽을 움직인 게 아니다.4-5가 이를 수치로 확인한다 — 해당 구간을 빼면 금액→매출 관계는 사라지고,단가→마진 관계만 남는다.### 동성화인텍 — 판단 보류재무가 2019Q4부터라 YoY 기준 n=23, lag 4에서 n=19다. 관측 구간에 LNG 발주 사이클이한 번밖에 들어오지 않아 어떤 상관도 사이클 1회 표본이다. 선행지표 채택 여부를판단할 근거가 없다.

In [ ]:
# 4-5. 강건성 — 2021Q4~2022Q2 구간 제외 시 관계가 유지되는가EXCL = pd.period_range("2021Q4", "2022Q2", freq="Q")sub = panel.drop(index=[p for p in EXCL if p in panel.index])print("■ 전체 구간 vs 2021Q4~2022Q2 제외")for xcol, ycol, lag, xlab, ylab in CANDIDATES:    for tag, pnl in [("전체", panel), ("제외", sub)]:        d = pd.concat([pnl[xcol].shift(lag).rename("x"), pnl[ycol].rename("y")], axis=1).dropna()        r, p = stats.pearsonr(d["x"], d["y"])        rs, ps = stats.spearmanr(d["x"], d["y"])        print(f"  {xlab:24s} -> {ylab:22s} [{tag}] n={len(d):2d}  "              f"r={r:+.3f} (p={p:.4f})  rho={rs:+.3f} (p={ps:.4f})")    print()

## 4-6. 실전 적용 지침1. **한국카본 마진 점검에 HS 390931 단가 YoY를 동행 지표로 쓴다.** 이 분석에서   구간 제외 검정을 통과한 유일한 관계다(r −0.574 → −0.572). 월별 MDI 수출단가는   분기 실적 공시보다 훨씬 빨리 나오므로, **발표 전 마진 방향을 가늠하는 용도**로는   실질적인 선행성이 있다. 통계적 lag는 0이지만 데이터 가용 시점이 앞선다.2. **HS 390931 수출금액의 3분기 선행 관계는 채택하지 않는다.** 전체 구간 r=−0.600이   특정 3개 분기(2021Q4~2022Q2)에 의존하며, 제외 시 p=0.063으로 무너진다.3. **물량(중량) 지표는 쓰지 않는다.** 두 회사 어느 타깃에서도 유의하지 않다.4. **HS 390950은 쓰지 않는다.** 산출물 코드이지만 두 회사 매출이 국내 조선소향이라   수출 통계에 실리지 않고, log 비율 추세가 연 +17.6~20.8%로 발산한다(R2 0.69~0.75).5. **동성화인텍에는 적용하지 않는다.** 표본 부족.6. 재실행 시 `END_Q`를 수출 원천 최신월이 속한 **직전 완결 분기**로 갱신할 것.   3개월 미만 분기는 `load_export_quarterly`가 자동 제외한다.